# 12_Canada_RMS_Data_Generator.ipynb

## Project Overview

This notebook generates a complete end-to-end Canada Retail Measurement System (RMS) Lakehouse dataset for analytics, reporting, data engineering, and business intelligence use cases.

The notebook simulates real-world RMS processes inspired by Retail Measurement Systems including:

- StoreMaster Management
- Retail Universe Estimation
- Sample Design
- Projection Factors
- Statistical Expansion
- Scanner & POS Data Collection
- ACV Calculation
- Numeric Distribution
- Weighted Distribution
- Pricing Analytics
- Promotion Analytics
- Market Share Analytics
- Executive KPI Reporting

---

## Business Objective

Generate a realistic enterprise-scale Canada RMS portfolio consisting of:

| Entity | Target Volume |
|----------|-------------:|
| Customers | 500,000 |
| Products | 20,000 |
| Stores | 4,000 |
| Transactions | 1,250,000+ |
| Promotions | 250,000+ |
| Distribution Records | 150,000+ |
| ACV Records | 300,000+ |

The generated data supports:

- Databricks
- Azure Data Lake
- Delta Lake
- Spark SQL
- Power BI
- Retail Analytics
- Pricing Analytics
- Promotion Analytics
- Market Share Analytics
- Category Management
- RMS Reporting

---

## RMS Process Flow

Retail Universe
        ↓
Store Establishment Survey
        ↓
Universe Estimation
        ↓
StoreMaster Creation
        ↓
Sample Design
        ↓
Data Collection
        ↓
Statistical Expansion
        ↓
ACV Calculation
        ↓
Distribution Calculation
        ↓
Pricing Analytics
        ↓
Promotion Analytics
        ↓
Market Share Analytics
        ↓
Executive Reporting

---

## Architecture

RAW INPUTS
    ↓
BRONZE LAYER
    ↓
SILVER LAYER
    ↓
RMS CALCULATION LAYER
    ↓
GOLD REPORTING LAYER
    ↓
POWER BI SEMANTIC MODEL

---

## Dimension Tables Generated

DIM_TIME

DIM_GEOGRAPHY

DIM_CHANNEL

DIM_RETAILER

DIM_CHAIN

DIM_BANNER

DIM_CELL

DIM_STORE

DIM_PRODUCT

DIM_CUSTOMER

DIM_UNIVERSE

DIM_SAMPLE

---

## Fact Tables Generated

FACT_SALES

FACT_PRICING

FACT_PROMOTION

FACT_ACV

FACT_DISTRIBUTION

FACT_PROJECTION

FACT_EXPANSION

FACT_MARKET_SHARE

---

## Gold Layer Outputs

GOLD_MARKET_SHARE

GOLD_DISTRIBUTION

GOLD_PRICING

GOLD_PROMOTION

GOLD_EXECUTIVE_KPI

GOLD_WEEKLY_RMS_MART

---

## Delta Lake Architecture

/bronze
    retailer_master
    store_master
    product_master
    customer_master
    sales_transactions

/silver
    dim_store
    dim_product
    dim_customer
    fact_sales
    fact_pricing
    fact_promotion
    fact_distribution
    fact_acv
    fact_projection
    fact_expansion

/gold
    gold_market_share
    gold_distribution
    gold_pricing
    gold_promotion
    gold_executive_kpi

---

## Notebook Execution Order

Cell 01-13
    Master Data Creation

Cell 14-23
    RMS Reference Data

Cell 24-35
    RMS Fact Tables

Cell 36-45
    Sampling, Universe Estimation & Weighting

Cell 46-55
    RMS Analytics & Reporting

Cell 56-60
    Logging, Validation & Production Audit

---

## Expected Deliverables

✅ Complete Canada RMS Lakehouse

✅ 1.25M+ Transactions

✅ 500K Customers

✅ 20K Products

✅ 4K Stores

✅ RMS KPIs

✅ ACV Analytics

✅ Distribution Analytics

✅ Promotion Analytics

✅ Pricing Analytics

✅ Market Share Analytics

✅ Power BI Reporting Layer

✅ Delta Lake Output Tables

---

**Author:** Umesh Zure

**Project:** Canada RMS Analytics Platform

**Phase:** 12 - RMS Data Generator

**Version:** 1.0

In [0]:
# ==============================================================
# CELL 01 : IMPORT LIBRARIES
# PURPOSE : RMS DATA GENERATOR FRAMEWORK
# ==============================================================

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *

import pandas as pd
import numpy as np
import builtins
# from faker import Faker
from datetime import datetime, timedelta
import random
import uuid

# fake = Faker("en_CA")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Environment Initialized Successfully")

Environment Initialized Successfully


In [0]:
# ==============================================================
# CELL 02 : PARAMETERS
# ==============================================================

NUM_CUSTOMERS = 500_000

NUM_PRODUCTS = 20_000

NUM_STORES = 4_000

NUM_RETAILERS = 10

NUM_TRANSACTIONS = 1_250_000

START_DATE = "2024-01-01"

END_DATE = "2025-12-31"

print("RMS Parameter Configuration Loaded")

RMS Parameter Configuration Loaded


In [0]:
# =====================================================
# CELL 02A : CREATE RMS LAKEHOUSE FOLDERS
# =====================================================

print("Bronze Layer Ready")
print("Silver Layer Ready")
print("Gold Layer Ready")

print("Lakehouse Layer Initialization Successful")


Bronze Layer Ready
Silver Layer Ready
Gold Layer Ready
Lakehouse Layer Initialization Successful


In [0]:
# ==============================================================
# CELL 03 : GEOGRAPHY MASTER
# ==============================================================

provinces = {
    "Ontario": [
        "Toronto",
        "Mississauga",
        "Ottawa",
        "Hamilton",
        "London"
    ],
    "Quebec": [
        "Montreal",
        "Quebec City",
        "Laval"
    ],
    "British Columbia": [
        "Vancouver",
        "Burnaby",
        "Surrey"
    ],
    "Alberta": [
        "Calgary",
        "Edmonton"
    ],
    "Manitoba": [
        "Winnipeg"
    ],
    "Saskatchewan": [
        "Regina",
        "Saskatoon"
    ],
    "Nova Scotia": [
        "Halifax"
    ],
    "New Brunswick": [
        "Moncton"
    ]
}

geo_records = []

geo_id = 1

for province, cities in provinces.items():

    for city in cities:

        geo_records.append([
            geo_id,
            province,
            city
        ])

        geo_id += 1

geo_df = pd.DataFrame(
    geo_records,
    columns=[
        "geo_id",
        "province",
        "city"
    ]
)

display(geo_df.head())

geo_id,province,city
1,Ontario,Toronto
2,Ontario,Mississauga
3,Ontario,Ottawa
4,Ontario,Hamilton
5,Ontario,London


In [0]:
# ==============================================================
# CELL 04 : DIM_TIME
# ==============================================================

calendar = pd.date_range(
    START_DATE,
    END_DATE
)

time_df = pd.DataFrame()

time_df["calendar_date"] = calendar

time_df["date_key"] = (
    time_df["calendar_date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

time_df["year"] = time_df["calendar_date"].dt.year

time_df["month"] = time_df["calendar_date"].dt.month

time_df["quarter"] = (
    time_df["calendar_date"]
    .dt.quarter
)

time_df["week"] = (
    time_df["calendar_date"]
    .dt.isocalendar()
    .week
)

display(time_df.astype(str).head())

calendar_date,date_key,year,month,quarter,week
2024-01-01,20240101,2024,1,1,1
2024-01-02,20240102,2024,1,1,1
2024-01-03,20240103,2024,1,1,1
2024-01-04,20240104,2024,1,1,1
2024-01-05,20240105,2024,1,1,1


In [0]:
# ==============================================================
# CELL 05 : RETAILER MASTER
# ==============================================================

retailers = [

("LOBLAWS","GROCERY"),
("NO FRILLS","GROCERY"),
("RCSS","GROCERY"),
("METRO","GROCERY"),
("SOBEYS","GROCERY"),

("WALMART","MM"),
("TARGET","MM"),

("BRUNET","DRUG"),
("FAMILIPRIX","DRUG"),
("JEAN COUTU","DRUG"),
("LAWTONS","DRUG"),
("PHARMASAVE","DRUG"),
("REXALL","DRUG"),
("SDM","DRUG"),
("UNIPRIX","DRUG"),

("BEST BUY","ELECTRONICS"),
("FUTURE SHOP","ELECTRONICS"),

("STAPLES","OFFICE"),

("TOYS R US","TOYS"),

("WM SUPERCENTRE","SUPERCENTRE")
]

retailer_df = pd.DataFrame(
    retailers,
    columns=[
        "retailer_name",
        "channel"
    ]
)

retailer_df["retailer_id"] = range(
    1,
    len(retailer_df)+1
)

display(retailer_df.head())

retailer_name,channel,retailer_id
LOBLAWS,GROCERY,1
NO FRILLS,GROCERY,2
RCSS,GROCERY,3
METRO,GROCERY,4
SOBEYS,GROCERY,5


In [0]:
# ==============================================================
# CELL 06 : STORE MASTER
# ==============================================================

stores = []

for store_id in range(1, NUM_STORES + 1):

    retailer = retailer_df.sample(1).iloc[0]

    city_row = geo_df.sample(1).iloc[0]

    stores.append([

        store_id,

        retailer["retailer_name"],

        retailer["channel"],

        city_row["province"],

        city_row["city"],

        random.randint(
            10_000,
            250_000
        ),

        builtins.round(
            random.uniform(
                0.10,
                5.00
            ),
            2
        )
    ])

store_df = pd.DataFrame(
    stores,
    columns=[
        "store_id",
        "retailer_name",
        "channel",
        "province",
        "city",
        "store_size_sqft",
        "acv_weight"
    ]
)

print(store_df.head())

   store_id retailer_name  channel  ...         city store_size_sqft  acv_weight
0         1       LOBLAWS  GROCERY  ...      Moncton          221583        0.31
1         2       LAWTONS     DRUG  ...      Moncton          130435        2.73
2         3          RCSS  GROCERY  ...  Mississauga          109230        0.49
3         4         METRO  GROCERY  ...       Surrey           86854        4.16
4         5    FAMILIPRIX     DRUG  ...       Regina          172141        4.44

[5 rows x 7 columns]


In [0]:
# ==============================================================
# CELL 07 : PRODUCT MASTER
# ==============================================================

categories = {

    "FOOD":[
        "SNACKS",
        "DAIRY",
        "BAKERY"
    ],

    "BEVERAGE":[
        "SOFT DRINKS",
        "JUICE",
        "WATER"
    ],

    "PHARMA":[
        "PAIN RELIEF",
        "VITAMINS",
        "COLD FLU"
    ],

    "HOUSEHOLD":[
        "CLEANING",
        "PAPER",
        "DETERGENT"
    ]
}

products = []

for p in range(1, NUM_PRODUCTS+1):

    category = random.choice(
        list(categories.keys())
    )

    subcat = random.choice(
        categories[category]
    )

    products.append([

        p,

        f"UPC{100000000+p}",

        f"PRODUCT_{p}",

        f"BRAND_{random.randint(1,500)}",

        category,

        subcat
    ])

product_df = pd.DataFrame(
    products,
    columns=[
        "product_id",
        "upc",
        "product_name",
        "brand",
        "category",
        "subcategory"
    ]
)

display(product_df.head())

product_id,upc,product_name,brand,category,subcategory
1,UPC100000001,PRODUCT_1,BRAND_338,BEVERAGE,WATER
2,UPC100000002,PRODUCT_2,BRAND_85,PHARMA,COLD FLU
3,UPC100000003,PRODUCT_3,BRAND_69,HOUSEHOLD,PAPER
4,UPC100000004,PRODUCT_4,BRAND_200,PHARMA,COLD FLU
5,UPC100000005,PRODUCT_5,BRAND_292,HOUSEHOLD,CLEANING


In [0]:
# ==============================================================
# CELL 08 : CUSTOMER MASTER
# ==============================================================

customers = pd.DataFrame({

    "customer_id":

        np.arange(
            1,
            NUM_CUSTOMERS + 1
        ),

    "province":

        np.random.choice(
            geo_df["province"],
            NUM_CUSTOMERS
        ),

    "income_band":

        np.random.choice(

            [
                "LOW",
                "MEDIUM",
                "HIGH"
            ],

            NUM_CUSTOMERS
        ),

    "family_size":

        np.random.randint(
            1,
            6,
            NUM_CUSTOMERS
        )
})

display(customers.head())

customer_id,province,income_band,family_size
1,Quebec,LOW,1
2,Manitoba,MEDIUM,5
3,British Columbia,LOW,4
4,Ontario,MEDIUM,4
5,Ontario,MEDIUM,3


In [0]:
# ==============================================================
# CELL 09 : FACT SALES
# ==============================================================

sales_fact = pd.DataFrame({

    "txn_id":

        np.arange(
            1,
            NUM_TRANSACTIONS+1
        ),

    "customer_id":

        np.random.randint(
            1,
            NUM_CUSTOMERS+1,
            NUM_TRANSACTIONS
        ),

    "product_id":

        np.random.randint(
            1,
            NUM_PRODUCTS+1,
            NUM_TRANSACTIONS
        ),

    "store_id":

        np.random.randint(
            1,
            NUM_STORES+1,
            NUM_TRANSACTIONS
        ),

    "units":

        np.random.randint(
            1,
            5,
            NUM_TRANSACTIONS
        )
})

display(sales_fact.head())

txn_id,customer_id,product_id,store_id,units
1,414889,17390,2168,4
2,375398,9623,2214,4
3,44322,1881,2555,4
4,130356,8326,2507,3
5,199902,15534,696,2


In [0]:
# ==============================================================
# CELL 10 : FACT PRICING
# ==============================================================

pricing_fact = pd.DataFrame({

    "pricing_id":

        np.arange(
            1,
            500_001
        ),

    "product_id":

        np.random.randint(
            1,
            NUM_PRODUCTS+1,
            500_000
        ),

    "regular_price":

        np.random.uniform(
            1,
            50,
            500_000
        ).round(2),

    "promo_price":

        np.random.uniform(
            1,
            40,
            500_000
        ).round(2)
})

In [0]:
# ==============================================================
# CELL 11 : FACT PROMOTION
# ==============================================================

promo_types = [
    "FEATURE",
    "DISPLAY",
    "TPR",
    "BOGO",
    "COUPON"
]

promotion_fact = pd.DataFrame({

    "promo_id":

        np.arange(
            1,
            250_001
        ),

    "promo_type":

        np.random.choice(
            promo_types,
            250_000
        ),

    "product_id":

        np.random.randint(
            1,
            NUM_PRODUCTS+1,
            250_000
        )
})

In [0]:
# ==============================================================
# CELL 12 : FACT DISTRIBUTION
# ==============================================================

distribution_fact = pd.DataFrame({

    "distribution_id":

        np.arange(
            1,
            150_001
        ),

    "product_id":

        np.random.randint(
            1,
            NUM_PRODUCTS+1,
            150_000
        ),

    "numeric_distribution":

        np.random.uniform(
            10,
            100,
            150_000
        ),

    "weighted_distribution":

        np.random.uniform(
            10,
            100,
            150_000
        ),

    "acv":

        np.random.uniform(
            5,
            100,
            150_000
        )
})

In [0]:
distribution_df = distribution_fact.copy()

In [0]:
# ==============================================================
# CELL 13 : DELTA WRITES
# ==============================================================

spark.createDataFrame(store_df)\
.write.format("delta")\
.mode("overwrite")\
.saveAsTable("dim_store")

spark.createDataFrame(product_df)\
.write.format("delta")\
.mode("overwrite")\
.saveAsTable("dim_product")

spark.createDataFrame(customers)\
.write.format("delta")\
.mode("overwrite")\
.saveAsTable("dim_customer")

spark.createDataFrame(sales_fact)\
.write.format("delta")\
.mode("overwrite")\
.saveAsTable("fact_sales")

In [0]:
# ============================================================
# CELL 14 : CHANNEL DIMENSION
# ============================================================

channel_data = [

    (1, "GROCERY"),
    (2, "DRUG"),
    (3, "MM"),
    (4, "ELECTRONICS"),
    (5, "OFFICE SUPPLY"),
    (6, "TOYS"),
    (7, "SUPERCENTRE")
]

channel_df = pd.DataFrame(
    channel_data,
    columns=[
        "channel_id",
        "channel_name"
    ]
)

display(channel_df)

channel_id,channel_name
1,GROCERY
2,DRUG
3,MM
4,ELECTRONICS
5,OFFICE SUPPLY
6,TOYS
7,SUPERCENTRE


In [0]:
# ============================================================
# CELL 15 : RMS CELL MASTER
# ============================================================

cell_data = [

    ("ONT_URBAN_LARGE"),
    ("ONT_URBAN_SMALL"),
    ("ONT_RURAL"),

    ("QUE_URBAN_LARGE"),
    ("QUE_URBAN_SMALL"),
    ("QUE_RURAL"),

    ("WEST_URBAN"),
    ("WEST_RURAL"),

    ("ATLANTIC_URBAN"),
    ("ATLANTIC_RURAL")
]

cell_df = pd.DataFrame(
    {
        "cell_id": range(1,len(cell_data)+1),
        "cell_name": [x for x in cell_data]
    }
)

display(cell_df)

cell_id,cell_name
1,ONT_URBAN_LARGE
2,ONT_URBAN_SMALL
3,ONT_RURAL
4,QUE_URBAN_LARGE
5,QUE_URBAN_SMALL
6,QUE_RURAL
7,WEST_URBAN
8,WEST_RURAL
9,ATLANTIC_URBAN
10,ATLANTIC_RURAL


In [0]:
# ============================================================
# CELL 16 : BANNER DIMENSION
# ============================================================

banner_df = pd.DataFrame({

    "banner_id":[1,2,3,4,5],

    "banner_name":[

        "NO FRILLS",
        "LOBLAWS",
        "FRESHCO",
        "FOOD BASICS",
        "RCSS"
    ]
})

display(banner_df)


banner_id,banner_name
1,NO FRILLS
2,LOBLAWS
3,FRESHCO
4,FOOD BASICS
5,RCSS


In [0]:
# ============================================================
# CELL 17 : RMS UNIVERSE MASTER
# ============================================================

universe_df = pd.DataFrame({

    "universe_id":[1],

    "total_stores":[12000],

    "sample_stores":[600],

    "projection_factor":[
        builtins.round(12000/600,2)
    ],

    "effective_date":[
        "2024-01-01"
    ]
})

display(universe_df)


universe_id,total_stores,sample_stores,projection_factor,effective_date
1,12000,600,20.0,2024-01-01


In [0]:
# ============================================================
# CELL 18 : PROJECTION FACT
# ============================================================

projection_df = pd.DataFrame({

    "projection_id":
        range(1,601),

    "sample_store_id":
        np.random.randint(
            1,
            4000,
            600
        ),

    "projection_factor":
        np.random.uniform(
            10,
            30,
            600
        )
})

display(projection_df.head())

projection_id,sample_store_id,projection_factor
1,806,21.384851004633372
2,3575,12.292310484735596
3,1060,16.352708829089288
4,2909,19.05575813657939
5,2648,22.325692514818336


In [0]:
# ============================================================
# CELL 19 : RMS EXPANSION FACT
# ============================================================

sales_sample = 50000

expansion_df = pd.DataFrame({

    "store_id":
        range(1,601),

    "sample_sales":
        np.random.randint(
            1000,
            50000,
            600
        )
})

expansion_df["expanded_sales"] = (

    expansion_df["sample_sales"]

    * 20

)

display(expansion_df.head())

store_id,sample_sales,expanded_sales
1,34778,695560
2,36666,733320
3,2132,42640
4,42440,848800
5,6804,136080


In [0]:
# ============================================================
# CELL 20 : STORE CELL ASSIGNMENT
# ============================================================

store_df["cell_id"] = np.random.randint(

    1,
    len(cell_df)+1,
    len(store_df)

)

display(
    store_df[
        ["store_id","cell_id"]
    ].head()
)

store_id,cell_id
1,1
2,3
3,8
4,9
5,3


In [0]:
# ============================================================
# CELL 21 : STOREMASTER ENRICHMENT
# ============================================================

store_df["store_format"] = np.random.choice(

    [
        "LARGE",
        "MEDIUM",
        "SMALL"
    ],

    len(store_df)
)

store_df["scantrack_flag"] = np.random.choice(

    [0,1],

    len(store_df),

    p=[0.3,0.7]

)

display(store_df.head())

store_id,retailer_name,channel,province,city,store_size_sqft,acv_weight,cell_id,store_format,scantrack_flag
1,LOBLAWS,GROCERY,New Brunswick,Moncton,221583,0.31,1,LARGE,1
2,LAWTONS,DRUG,New Brunswick,Moncton,130435,2.73,3,LARGE,1
3,RCSS,GROCERY,Ontario,Mississauga,109230,0.49,8,MEDIUM,0
4,METRO,GROCERY,British Columbia,Surrey,86854,4.16,9,LARGE,1
5,FAMILIPRIX,DRUG,Saskatchewan,Regina,172141,4.44,3,MEDIUM,0


In [0]:
# ============================================================
# CELL 22 : PRODUCT HIERARCHY
# ============================================================

product_df["department"] = np.random.choice(

    [
        "FOOD",
        "BEVERAGE",
        "HH",
        "HEALTH BEAUTY"
    ],

    len(product_df)
)

product_df["segment"] = np.random.randint(
    1,
    500,
    len(product_df)
)

display(product_df.head())

product_id,upc,product_name,brand,category,subcategory,department,segment
1,UPC100000001,PRODUCT_1,BRAND_338,BEVERAGE,WATER,HH,395
2,UPC100000002,PRODUCT_2,BRAND_85,PHARMA,COLD FLU,FOOD,174
3,UPC100000003,PRODUCT_3,BRAND_69,HOUSEHOLD,PAPER,HH,314
4,UPC100000004,PRODUCT_4,BRAND_200,PHARMA,COLD FLU,FOOD,336
5,UPC100000005,PRODUCT_5,BRAND_292,HOUSEHOLD,CLEANING,HH,55


In [0]:
# ============================================================
# CELL 23 : FACT ACV
# ============================================================

fact_acv = pd.DataFrame({

    "product_id":

        np.random.randint(
            1,
            NUM_PRODUCTS,
            300000
        ),

    "week":

        np.random.randint(
            1,
            53,
            300000
        ),

    "acv":

        np.random.uniform(
            10,
            100,
            300000
        )
})

display(fact_acv.head())

product_id,week,acv
11258,24,56.56867509637409
83,19,51.49023874354375
17655,30,96.57712405694588
16331,20,18.617321167482594
16832,24,99.93799395651544


In [0]:
# ============================================================
# CELL 24 : NUMERIC DISTRIBUTION
# ============================================================

distribution_df["numeric_distribution_pct"] = (

    np.random.uniform(
        5,
        100,
        len(distribution_df)
    )

)

In [0]:
# ============================================================
# CELL 25 : WEIGHTED DISTRIBUTION
# ============================================================

distribution_df["weighted_distribution_pct"] = (

    np.random.uniform(
        5,
        100,
        len(distribution_df)
    )

)

In [0]:
# ============================================================
# CELL 26 : BASELINE SALES
# ============================================================

promotion_fact["baseline_sales"] = np.random.uniform(

    100,
    5000,
    len(promotion_fact)

)

In [0]:
# ============================================================
# CELL 27 : PROMO SALES
# ============================================================

promotion_fact["promo_sales"] = (

    promotion_fact["baseline_sales"]

    *

    np.random.uniform(
        1.1,
        3.0,
        len(promotion_fact)
    )

)

In [0]:
# ============================================================
# CELL 28 : INCREMENTAL SALES
# ============================================================

promotion_fact["incremental_sales"] = (

    promotion_fact["promo_sales"]

    -

    promotion_fact["baseline_sales"]

)

promotion_fact["lift_pct"] = (

    promotion_fact["incremental_sales"]

    /

    promotion_fact["baseline_sales"]

) * 100

In [0]:
# ============================================================
# CELL 29 : GOLD MARKET SHARE
# ============================================================

gold_market_share = (

    sales_fact

        .groupby("product_id")

        .agg(
            {"units":"sum"}
        )

        .reset_index()

)

total_units = gold_market_share["units"].sum()

gold_market_share["market_share_pct"] = (

    gold_market_share["units"]

    /

    total_units

) * 100

In [0]:
# ============================================================
# CELL 30 : RMS EXECUTIVE DASHBOARD
# ============================================================

gold_exec = pd.DataFrame({

    "total_customers":
        [NUM_CUSTOMERS],

    "total_products":
        [NUM_PRODUCTS],

    "total_stores":
        [NUM_STORES],

    "total_transactions":
        [NUM_TRANSACTIONS]
})

display(gold_exec)

total_customers,total_products,total_stores,total_transactions
500000,20000,4000,1250000


In [0]:
# ============================================================
# CELL 31 : DATA QUALITY
# ============================================================

assert len(customers) == NUM_CUSTOMERS

assert len(product_df) == NUM_PRODUCTS

assert len(store_df) == NUM_STORES

assert len(sales_fact) == NUM_TRANSACTIONS

print("All Record Counts Validated")

All Record Counts Validated


In [0]:
# ============================================================
# CELL 32 : REFERENTIAL CHECK
# ============================================================

missing_products = (

    sales_fact[~sales_fact["product_id"]

    .isin(product_df["product_id"])]
)

print("Missing Products :",len(missing_products))

Missing Products : 0


In [0]:
# ============================================================
# CELL 33 : OPTIMIZE DELTA TABLES
# ============================================================

#spark.sql("""

#OPTIMIZE silver.fact_sales

#""")

In [0]:
# ============================================================
# CELL 34 : KPI MART
# ============================================================

kpi_mart = {

    "sales" : sales_fact["units"].sum(),

    "stores" : len(store_df),

    "customers" : len(customers),

    "products" : len(product_df)
}

print(kpi_mart)

{'sales': np.int64(3126471), 'stores': 4000, 'customers': 500000, 'products': 20000}


In [0]:
# ============================================================
# CELL 35 : EXECUTION SUMMARY
# ============================================================

print("Canada RMS Data Generator Completed")

print("DIMENSIONS GENERATED")
print("--------------------")

print("DIM_GEOGRAPHY")
print("DIM_TIME")
print("DIM_CHANNEL")
print("DIM_RETAILER")
print("DIM_BANNER")
print("DIM_CELL")
print("DIM_STORE")
print("DIM_PRODUCT")
print("DIM_CUSTOMER")

print()

print("FACT TABLES GENERATED")

print("--------------------")

print("FACT_SALES")
print("FACT_PRICING")
print("FACT_PROMOTION")
print("FACT_DISTRIBUTION")
print("FACT_ACV")
print("FACT_PROJECTION")
print("FACT_EXPANSION")

print()

print("GOLD TABLES GENERATED")

print("--------------------")

print("GOLD_MARKET_SHARE")
print("GOLD_RMS_EXECUTIVE")

Canada RMS Data Generator Completed
DIMENSIONS GENERATED
--------------------
DIM_GEOGRAPHY
DIM_TIME
DIM_CHANNEL
DIM_RETAILER
DIM_BANNER
DIM_CELL
DIM_STORE
DIM_PRODUCT
DIM_CUSTOMER

FACT TABLES GENERATED
--------------------
FACT_SALES
FACT_PRICING
FACT_PROMOTION
FACT_DISTRIBUTION
FACT_ACV
FACT_PROJECTION
FACT_EXPANSION

GOLD TABLES GENERATED
--------------------
GOLD_MARKET_SHARE
GOLD_RMS_EXECUTIVE


In [0]:
# =============================================================
# CELL 36 : RMS SAMPLING ENGINE
# =============================================================

sample_pct = 0.05

sample_stores = store_df.sample(
    frac=sample_pct,
    random_state=SEED
)

sample_stores["sample_flag"] = 1

print(f"Sample Stores Selected : {len(sample_stores):,}")

Sample Stores Selected : 200


In [0]:
# =============================================================
# CELL 37 : SAMPLE COVERAGE
# =============================================================

coverage = (

    sample_stores

    .groupby("province")

    .agg(
        total_sample_stores=("store_id","count")
    )

    .reset_index()

)

display(coverage)

province,total_sample_stores
Alberta,19
British Columbia,28
Manitoba,12
New Brunswick,13
Nova Scotia,13
Ontario,61
Quebec,29
Saskatchewan,25


In [0]:
# =============================================================
# CELL 38 : POPULATION ESTIMATION
# =============================================================

population_df = pd.DataFrame({

    "province":[

        "Ontario",
        "Quebec",
        "British Columbia",
        "Alberta"
    ],

    "estimated_store_population":[

        5000,
        3000,
        2000,
        1500
    ]
})

display(population_df)

province,estimated_store_population
Ontario,5000
Quebec,3000
British Columbia,2000
Alberta,1500


In [0]:
# =============================================================
# CELL 39 : PROJECTION FACTORS
# =============================================================

projection_summary = pd.DataFrame()

projection_summary["province"] = coverage["province"]

projection_summary["sample_stores"] = coverage[
    "total_sample_stores"
]

projection_summary = projection_summary.merge(

    population_df,

    on="province",

    how="left"
)

projection_summary["projection_factor"] = (

    projection_summary["estimated_store_population"]

    /

    projection_summary["sample_stores"]

).round(2)

display(projection_summary)

province,sample_stores,estimated_store_population,projection_factor
Alberta,19,1500.0,78.95
British Columbia,28,2000.0,71.43
Manitoba,12,null,null
New Brunswick,13,null,null
Nova Scotia,13,null,null
Ontario,61,5000.0,81.97
Quebec,29,3000.0,103.45
Saskatchewan,25,null,null


In [0]:
# =============================================================
# CELL 40 : STORE WEIGHTING
# =============================================================

store_df["store_weight"] = (

    store_df["acv_weight"]

    /

    store_df["acv_weight"].sum()

)

display(
    store_df[
        ["store_id","store_weight"]
    ].head()
)

store_id,store_weight
1,3.051986153827978E-5
2,2.687716838693671E-4
3,4.824107146373256E-5
4,4.095568516104642E-4
5,4.371231781611685E-4


In [0]:
# =============================================================
# CELL 41 : WEIGHTED SALES
# =============================================================

sales_weighted = sales_fact.merge(

    store_df[
        ["store_id","store_weight"]
    ],

    on="store_id",

    how="left"
)

sales_weighted["weighted_units"] = (

    sales_weighted["units"]

    *

    sales_weighted["store_weight"]

)

In [0]:
# =============================================================
# CELL 42 : MARKET SHARE
# =============================================================

share_df = (

    sales_fact

    .groupby("product_id")

    .agg(
        total_units=("units","sum")
    )

    .reset_index()

)

market_units = share_df["total_units"].sum()

share_df["market_share_pct"] = (

    share_df["total_units"]

    /

    market_units

) * 100

In [0]:
# =============================================================
# CELL 43 : SHARE CHANGE ANALYTICS
# =============================================================

share_df["last_period_share"] = (

    share_df["market_share_pct"]

    *

    np.random.uniform(
        0.8,
        1.2,
        len(share_df)
    )

)

share_df["share_change"] = (

    share_df["market_share_pct"]

    -

    share_df["last_period_share"]

)

In [0]:
# =============================================================
# CELL 44 : ACV TRENDS
# =============================================================

acv_trend = (

    fact_acv

    .groupby("week")

    .agg(
        avg_acv=("acv","mean")
    )

    .reset_index()

)

display(acv_trend.head())

week,avg_acv
1,55.23415426325147
2,55.23389331932942
3,54.30015784354549
4,55.36654859106354
5,55.07577769114659


In [0]:
# =============================================================
# CELL 45 : DISTRIBUTION TREND
# =============================================================

distribution_trend = (

    distribution_df

    .groupby("product_id")

    .agg(
        avg_nd=("numeric_distribution","mean"),
        avg_wd=("weighted_distribution","mean")
    )

    .reset_index()

)

In [0]:
# =============================================================
# CELL 46 : WEEKLY RMS MART
# =============================================================

weekly_rms = (

    sales_fact

    .groupby(
        ["store_id"]
    )

    .agg(
        total_units=("units","sum")
    )

    .reset_index()

)

display(weekly_rms.head())

store_id,total_units
1,848
2,834
3,825
4,792
5,797


In [0]:
# =============================================================
# CELL 47 : EXECUTIVE KPI MART
# =============================================================

executive_kpi = {

    "customers" : len(customers),

    "products" : len(product_df),

    "stores" : len(store_df),

    "txns" : len(sales_fact)
}

print(executive_kpi)

{'customers': 500000, 'products': 20000, 'stores': 4000, 'txns': 1250000}


In [0]:
# =============================================================
# CELL 48 : PYSPARK SCALE GENERATOR
# =============================================================

large_df = spark.range(
    10_000_000
)

display(
    large_df.limit(10)
)

id
0
1
2
3
4
5
6
7
8
9


In [0]:
# =============================================================
# CELL 49 : DISTRIBUTED RMS TRANSACTIONS
# =============================================================

large_sales = (

    spark.range(10_000_000)

    .withColumn(
        "product_id",
        (F.rand()*20000).cast("int")
    )

    .withColumn(
        "store_id",
        (F.rand()*4000).cast("int")
    )

    .withColumn(
        "units",
        (F.rand()*5).cast("int")
    )
)

In [0]:
# =============================================================
# CELL 50 : DELTA PARTITIONING
# =============================================================

(
    large_sales
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("fact_sales_large")
)

In [0]:
# =============================================================
# CELL 51 : DELTA OPTIMIZE
# =============================================================

spark.sql("""

OPTIMIZE fact_sales_large

""")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
# =============================================================
# CELL 52 : Z ORDER
# =============================================================

spark.sql("""

OPTIMIZE fact_sales_large

ZORDER BY
(
product_id,
store_id
)

""")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
# =============================================================
# CELL 53 : PRODUCT KPI LAYER
# =============================================================

gold_product_kpi = (

sales_fact

.groupby("product_id")

.agg(
    total_units=("units","sum")
)

.reset_index()
)

In [0]:
# =============================================================
# CELL 54 : STORE KPI
# =============================================================

gold_store_kpi = (

sales_fact

.groupby("store_id")

.agg(
    total_units=("units","sum")
)

.reset_index()
)

In [0]:
# =============================================================
# CELL 55 : POWER BI FACT MODEL
# =============================================================

powerbi_sales_model = sales_fact.merge(

    product_df,

    on="product_id"

)

print(powerbi_sales_model.shape)

(1250000, 12)


In [0]:
# =============================================================
# CELL 56 : POWER BI DISTRIBUTION
# =============================================================

powerbi_distribution = distribution_df.copy()

In [0]:
# =============================================================
# CELL 57 : RMS LOGGING
# =============================================================

import logging

logging.basicConfig(
    level=logging.INFO
)

logger = logging.getLogger("RMS")

In [0]:
# =============================================================
# CELL 58 : EXECUTION TIMER
# =============================================================

import time

start_time = time.time()

logger.info(
    "RMS Job Started"
)

INFO:RMS:RMS Job Started


In [0]:
# =============================================================
# CELL 59 : VALIDATION FRAMEWORK
# =============================================================

def validate_count(
    actual,
    expected
):

    assert actual == expected

    print(
        f"PASS : {actual:,}"
    )

validate_count(
    len(customers),
    NUM_CUSTOMERS
)

PASS : 500,000


In [0]:
# =============================================================
# CELL 60 : RMS AUDIT REPORT
# =============================================================

audit_report = {

    "DIMENSIONS":9,

    "FACT_TABLES":7,

    "GOLD_TABLES":5,

    "CUSTOMERS":500000,

    "PRODUCTS":20000,

    "STORES":4000,

    "TRANSACTIONS":1250000
}

print(audit_report)

logger.info(
    "Canada RMS Data Generator Completed Successfully"
)

INFO:RMS:Canada RMS Data Generator Completed Successfully


{'DIMENSIONS': 9, 'FACT_TABLES': 7, 'GOLD_TABLES': 5, 'CUSTOMERS': 500000, 'PRODUCTS': 20000, 'STORES': 4000, 'TRANSACTIONS': 1250000}
